# ClaimLens ML Microservice: Inpatient Cost Exploration & Model Training

This notebook covers:
1. **Exploratory Data Analysis (EDA)** on ~10,000 synthetic Indian hospital tariff records.
2. **Data Preprocessing & Stratified Train-Test Split**.
3. **Quantile LightGBM Regressor Training** (alpha = 0.10, 0.50, 0.90).
4. **Model Evaluation** via Pinball Loss and Coverage Probability.
5. **Explainability & SHAP Analysis** (Summary Plot & Waterfall Plot).
6. **Artifact Export** to models/ directory.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path to import generate_dataset
sys.path.append('..')
from data.generate_dataset import generate_dataset

csv_path = '../data/hospital_tariffs.csv'
if not os.path.exists(csv_path):
    df = generate_dataset(num_records=10000, seed=42, output_path=csv_path)
else:
    df = pd.read_csv(csv_path)

print("Dataset Shape:", df.shape)
print("\nFirst 5 Records:")
display(df.head())

print("\nDescriptive Statistics:")
display(df.describe())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

FEATURE_COLS = ['procedure_name', 'patient_age', 'city_tier', 'hospital_tier', 'room_category', 'stay_duration_days']
CATEGORICAL_COLS = ['procedure_name', 'room_category']
NUMERICAL_COLS = ['patient_age', 'city_tier', 'hospital_tier', 'stay_duration_days']
TARGET_COL = 'total_bill'

X = df[FEATURE_COLS]
y = df[TARGET_COL]

# Stratified split based on procedure_name
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=df['procedure_name']
)

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_COLS),
        ('num', StandardScaler(), NUMERICAL_COLS)
    ],
    remainder='drop'
)

X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

feature_names = list(preprocessor.named_transformers_['cat'].get_feature_names_out(CATEGORICAL_COLS)) + NUMERICAL_COLS
print("Transformed Feature Matrix Shape:", X_train_trans.shape)
print("Total Encoded Features:", len(feature_names))

In [ ]:
from lightgbm import LGBMRegressor

alphas = [0.10, 0.50, 0.90]
models = {}

for alpha in alphas:
    key = f"p{int(alpha * 100)}"
    print(f"Training LightGBM Quantile Regressor for alpha={alpha:.2f}...")
    model = LGBMRegressor(
        objective='quantile',
        alpha=alpha,
        n_estimators=250,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        verbosity=-1
    )
    model.fit(X_train_trans, y_train)
    models[key] = model

print("All 3 quantile models successfully trained.")

In [ ]:
p10_preds = models['p10'].predict(X_test_trans)
p50_preds = models['p50'].predict(X_test_trans)
p90_preds = models['p90'].predict(X_test_trans)

def pinball_loss(y_true, y_pred, alpha):
    err = y_true - y_pred
    return np.mean(np.maximum(alpha * err, (alpha - 1) * err))

loss_p10 = pinball_loss(y_test.values, p10_preds, 0.10)
loss_p50 = pinball_loss(y_test.values, p50_preds, 0.50)
loss_p90 = pinball_loss(y_test.values, p90_preds, 0.90)

coverage = np.mean((y_test.values >= p10_preds) & (y_test.values <= p90_preds))

print("=== Quantile Model Evaluation ===")
print(f"Pinball Loss (P10): {loss_p10:.2f}")
print(f"Pinball Loss (P50 - Median): {loss_p50:.2f}")
print(f"Pinball Loss (P90): {loss_p90:.2f}")
print(f"Coverage Probability (P10 <= y <= P90): {coverage * 100:.2f}% (Target: ~80%)")

In [ ]:
import shap

explainer = shap.TreeExplainer(models['p50'])
shap_values = explainer(X_test_trans)
shap_values.feature_names = feature_names

plt.figure(figsize=(10, 6))
plt.title("SHAP Summary Plot for Inpatient Cost Estimation (P50 Model)")
shap.summary_plot(shap_values, X_test_trans, feature_names=feature_names, show=False)
plt.show()

sample_idx = 0
print("\nSample Patient Feature Inputs:")
display(X_test.iloc[[sample_idx]])
print("True Bill:", y_test.iloc[sample_idx], "Predicted P50:", p50_preds[sample_idx])

plt.figure(figsize=(8, 5))
shap.plots.waterfall(shap_values[sample_idx], show=False)
plt.title(f"SHAP Waterfall Explanation for Patient #{sample_idx}")
plt.show()

In [ ]:
import joblib

models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

joblib.dump(preprocessor, os.path.join(models_dir, 'preprocessor.pkl'))
joblib.dump(models['p10'], os.path.join(models_dir, 'cost_p10.pkl'))
joblib.dump(models['p50'], os.path.join(models_dir, 'cost_p50.pkl'))
joblib.dump(models['p90'], os.path.join(models_dir, 'cost_p90.pkl'))

print(f"Successfully exported preprocessor and 3 LightGBM quantile models to '{models_dir}/'")